# 第七章：生产实践（调试、追踪、部署）

## 学习目标

- 掌握 LangChain 的调试技巧（verbose、set_debug、回调）
- 理解 Callback 系统的原理与用法
- 使用 Pydantic 实现结构化输出（`with_structured_output`）
- 利用缓存减少重复 API 调用
- 实现重试与降级策略
- 了解速率限制、可观测性追踪与部署方案
- 回顾整个学习路线

---

## 0. 环境准备

In [5]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")
import langchain
print(f"langchain 版本: {langchain.__version__}")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
    temperature=0.7,
)
print("模型初始化完成")

langchain 版本: 1.2.13
模型初始化完成


---

## 1. 调试技巧

前六章我们构建了各种链和 Agent，但遇到问题时怎么排查？LangChain 提供了多层调试手段。

### 1.1 set_verbose / set_debug

In [9]:
from langchain_core.globals import set_verbose, set_debug
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# verbose: 打印链的输入输出
set_verbose(True)

chain = ChatPromptTemplate.from_template("用一句话解释{concept}") | llm | StrOutputParser()
result = chain.invoke({"concept": "递归"})
print(result)

set_verbose(False)

**递归**：函数调用自身来解决问题，每次调用都向基准条件（终止条件）逼近，将大问题分解为相似的更小问题。


In [7]:
# debug: 更详细，打印每个组件的输入输出和中间状态
set_debug(True)

result = chain.invoke({"concept": "哈希表"})
print(result)

set_debug(False)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "concept": "哈希表"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "concept": "哈希表"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: 用一句话解释哈希表"
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatOpenAI] [1.57s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "哈希表是通过**哈希函数将键映射到数组索引**，实现平均 O(1) 时间复杂度的键值存储数据结构。",
        "generation_info": {
          "finish_reason": "stop",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs"

两者的区别：

- **`set_verbose(True)`**：打印链的高层输入/输出，适合快速了解数据流向
- **`set_debug(True)`**：打印所有细节，包括完整的 prompt 文本、模型原始返回、每个组件的中间结果

| 模式 | 信息量 | 适用场景 |
|------|--------|----------|
| `set_verbose(True)` | 中等 | 日常开发，确认链的输入输出是否正确 |
| `set_debug(True)` | 非常详细 | 排查诡异问题，检查 prompt 是否按预期拼装 |
| 都关闭 | 无额外输出 | 生产环境 |

建议：开发阶段默认开 `verbose`，遇到问题时临时切到 `debug`，解决后关掉。

### 1.2 查看链的结构

当链变复杂时，可以可视化它的执行图，看清楚数据流经了哪些组件。

In [ ]:
# 打印链的 ASCII 结构图
chain.get_graph().print_ascii()

对于更复杂的链（比如包含 `RunnableParallel`、分支逻辑等），这个图能帮你快速定位数据在哪个节点出了问题。

---

## 2. Callback 系统

`set_verbose` 和 `set_debug` 是全局开关，粒度太粗。Callback 系统让你精确控制「在哪个环节做什么」——记录耗时、统计 token、写日志、发告警，都靠它。

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
import time

class TimingCallback(BaseCallbackHandler):
    """记录每次 LLM 调用的耗时"""

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.start_time = time.time()
        print(f"LLM 开始调用...")

    def on_llm_end(self, response, **kwargs):
        elapsed = time.time() - self.start_time
        print(f"LLM 调用完成，耗时 {elapsed:.2f} 秒")

    def on_llm_error(self, error, **kwargs):
        print(f"LLM 调用出错: {error}")

timing = TimingCallback()
result = chain.invoke({"concept": "协程"}, config={"callbacks": [timing]})
print(result)

Callback 通过 `config={"callbacks": [...]}` 传入，只对当前调用生效，不影响全局。

继承 `BaseCallbackHandler` 后，覆写你关心的方法即可——不需要实现所有方法。

In [ ]:
class TokenCounter(BaseCallbackHandler):
    """统计 token 使用量"""
    def __init__(self):
        self.total_tokens = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0

    def on_llm_end(self, response, **kwargs):
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage", {})
            self.prompt_tokens += usage.get("prompt_tokens", 0)
            self.completion_tokens += usage.get("completion_tokens", 0)
            self.total_tokens += usage.get("total_tokens", 0)

counter = TokenCounter()
results = chain.batch(
    [{"concept": "微服务"}, {"concept": "容器化"}, {"concept": "CI/CD"}],
    config={"callbacks": [counter]},
)
for r in results:
    print(r)
print(f"\n总 token: {counter.total_tokens}")
print(f"输入 token: {counter.prompt_tokens}")
print(f"输出 token: {counter.completion_tokens}")

Callback 还支持链和工具级别的事件。常用的方法一览：

| 方法 | 触发时机 |
|------|----------|
| `on_llm_start` | LLM 调用开始 |
| `on_llm_end` | LLM 调用结束 |
| `on_llm_error` | LLM 调用出错 |
| `on_chain_start` | Chain 开始执行 |
| `on_chain_end` | Chain 执行完成 |
| `on_tool_start` | 工具开始调用 |
| `on_tool_end` | 工具调用完成 |

### 常见问题

- **token 统计为 0**：部分模型 API 不在响应中返回 `token_usage` 信息，这取决于 API 提供方的实现。可以通过 `response.llm_output` 检查返回了哪些字段。

---

## 3. 结构化输出（`with_structured_output`）

前面几章中，我们用 `StrOutputParser` 拿到的都是纯文本。但实际项目中，你往往需要模型返回结构化数据（JSON、对象），以便后续程序处理。

第三章学过用 `JsonOutputParser` + prompt 指令来引导模型输出 JSON，但这种方式不够可靠——模型可能输出格式错误的 JSON。

`with_structured_output` 利用模型的 function calling 能力，强制输出符合指定 schema 的结构化数据。

In [ ]:
from pydantic import BaseModel, Field

class MovieReview(BaseModel):
    """电影评价的结构化输出"""
    title: str = Field(description="电影名称")
    rating: int = Field(description="评分 1-10")
    summary: str = Field(description="一句话评价")
    tags: list[str] = Field(description="标签列表")

structured_llm = llm.with_structured_output(MovieReview)
result = structured_llm.invoke("评价一下电影《盗梦空间》")
print(f"类型: {type(result)}")
print(f"电影: {result.title}")
print(f"评分: {result.rating}")
print(f"评价: {result.summary}")
print(f"标签: {result.tags}")

返回的直接是 `MovieReview` 对象，不是字符串。字段类型、字段数量都由 Pydantic model 约束，拿到的结果可以直接用，不需要手动解析 JSON。

与第三章 `JsonOutputParser` 的对比：

| 方式 | 原理 | 可靠性 |
|------|------|--------|
| `JsonOutputParser` | 在 prompt 中要求模型输出 JSON，再解析 | 模型可能输出格式有误 |
| `with_structured_output` | 利用 function calling / tool calling 机制 | 由 API 层面保证格式正确 |

In [ ]:
# 在链中使用结构化输出
class TopicAnalysis(BaseModel):
    """主题分析结果"""
    main_topic: str = Field(description="主要话题")
    sentiment: str = Field(description="情感倾向：正面/负面/中性")
    key_points: list[str] = Field(description="关键要点列表")
    confidence: float = Field(description="置信度 0-1")

analysis_chain = (
    ChatPromptTemplate.from_template("分析以下文本：\n{text}")
    | llm.with_structured_output(TopicAnalysis)
)

result = analysis_chain.invoke({
    "text": "LangChain 最近发布了新版本，标志着框架的成熟。社区反响热烈，开发者们期待已久。"
})
print(f"话题: {result.main_topic}")
print(f"情感: {result.sentiment}")
print(f"要点: {result.key_points}")
print(f"置信度: {result.confidence}")

`with_structured_output` 返回的是一个新的 Runnable，可以直接用 `|` 接在 prompt 后面，和 LCEL 无缝配合。

### 常见问题

- **报错 `NotImplementedError`**：说明当前模型不支持 function calling。需要确认使用的模型和 API 支持 tool/function calling 功能。
- **字段值不合理（如评分超出范围）**：Pydantic 只约束类型，不约束语义。需要在 `Field(description=...)` 中写清楚取值范围，或者加 Pydantic validator。

---

## 4. 缓存

相同的问题反复调 API，既慢又费钱。缓存让相同输入直接返回之前的结果。

In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
import time

set_llm_cache(InMemoryCache())

# 第一次调用：正常请求 API
start = time.time()
result1 = llm.invoke("什么是人工智能？")
time1 = time.time() - start
print(f"第一次调用: {time1:.2f}秒")
print(result1.content[:50])

# 第二次调用：命中缓存，瞬间返回
start = time.time()
result2 = llm.invoke("什么是人工智能？")
time2 = time.time() - start
print(f"\n第二次调用: {time2:.4f}秒")
print(result2.content[:50])
print(f"\n加速比: {time1/time2:.0f}x")

第二次调用几乎零延迟，因为输入完全相同，直接从内存缓存返回结果。

`InMemoryCache` 存在进程内存中，重启就没了。适合开发调试阶段。

In [ ]:
# SQLite 缓存：持久化到文件，重启后仍然有效
from langchain_community.cache import SQLiteCache

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))
result = llm.invoke("解释什么是云计算")
print(result.content[:80])
print("\nSQLite 缓存已启用，缓存文件: .langchain_cache.db")

In [ ]:
# 清理缓存
set_llm_cache(None)

if os.path.exists(".langchain_cache.db"):
    os.remove(".langchain_cache.db")
print("缓存已清理")

| 缓存类型 | 持久化 | 适用场景 |
|----------|--------|----------|
| `InMemoryCache` | 否（进程退出即丢失） | 开发调试、Notebook 实验 |
| `SQLiteCache` | 是（文件存储） | 单机生产环境 |
| `RedisCache` 等 | 是（远程存储） | 分布式多实例部署 |

### 常见问题

- **缓存导致结果不更新**：缓存是精确匹配输入的，哪怕多一个空格都不会命中。如果你修改了 prompt 但忘记清缓存，可能以为修改没生效。
- **不适合缓存的场景**：需要实时信息（如当前时间）、需要随机性的场景不应该开缓存。

---

## 5. 重试与降级

网络抖动、API 限流、服务暂时不可用——这些在生产环境中都是家常便饭。`with_retry` 和 `with_fallbacks` 帮你自动处理这些情况。

In [ ]:
# with_retry: 失败后自动重试，指数退避 + 随机抖动
reliable_llm = llm.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True,
)
result = reliable_llm.invoke("你好")
print(result.content)

`with_retry` 的行为：

- 第一次调用失败 → 等待短暂时间后重试
- 重试间隔指数增长（如 1s → 2s → 4s），加上随机抖动避免多个客户端同时重试（"惊群效应"）
- 达到 `stop_after_attempt` 次数后放弃，抛出原始异常

第三章学过 `with_fallbacks`，可以和重试组合使用：

In [ ]:
# with_fallbacks: 主模型失败后切换到备用模型
primary_llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)
fallback_llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

# 主模型失败 → 自动切换到备用模型
robust_llm = primary_llm.with_fallbacks([fallback_llm])
result = robust_llm.invoke("什么是量子计算？")
print(result.content)

生产中的推荐组合：

```python
# 重试 + 降级
production_llm = (
    primary_llm.with_retry(stop_after_attempt=2)
    .with_fallbacks([fallback_llm.with_retry(stop_after_attempt=2)])
)
```

主模型重试 2 次 → 仍然失败 → 切到备用模型 → 备用模型也重试 2 次 → 都失败才报错。

---

## 6. 速率限制（Rate Limiting）

API 通常有调用频率限制（如每秒最多 N 次请求）。批量调用时很容易触发限流，导致大量 429 错误。`InMemoryRateLimiter` 在客户端主动限速，避免被服务端拒绝。

In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=1,       # 每秒最多 1 个请求
    check_every_n_seconds=0.1,   # 检查间隔
    max_bucket_size=5,           # 令牌桶大小（允许短时间突发）
)

rate_limited_llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
    rate_limiter=rate_limiter,
)

start = time.time()
results = rate_limited_llm.batch(["1+1=?", "2+2=?", "3+3=?"])
elapsed = time.time() - start
for r in results:
    print(r.content)
print(f"\n耗时: {elapsed:.2f}秒（受速率限制）")

速率限制器使用令牌桶算法：桶里有令牌时立即放行，没有令牌时等待。`max_bucket_size` 控制桶的最大容量，允许一定程度的突发请求。

| 参数 | 说明 |
|------|------|
| `requests_per_second` | 平均请求速率 |
| `check_every_n_seconds` | 令牌刷新检查间隔 |
| `max_bucket_size` | 令牌桶大小，控制突发上限 |

---

## 7. 可观测性（LangSmith 简介）

调试和 Callback 能解决开发阶段的问题，但应用上线后呢？你需要一个持续追踪系统，记录每次调用的完整链路、耗时、token 用量、输入输出。

LangSmith 是 LangChain 官方提供的可观测性平台，最大的好处是**零代码侵入**——只需设置环境变量，所有 LangChain 调用自动上报。

In [ ]:
# LangSmith 追踪配置（设置环境变量即可启用）
print("LangSmith 追踪配置（设置环境变量即可启用）：")
print("  LANGCHAIN_TRACING_V2=true")
print("  LANGCHAIN_API_KEY=your-api-key")
print("  LANGCHAIN_PROJECT=my-project")
print()
print("启用后所有 LangChain 调用自动上报，无需修改代码")

LangSmith 提供的核心功能：

| 功能 | 说明 |
|------|------|
| Trace 追踪 | 每次调用的完整链路图，包括每个组件的输入输出和耗时 |
| Token 统计 | 自动统计每次调用的 token 消耗和成本 |
| 错误监控 | 记录失败的调用，方便排查线上问题 |
| 数据集与评估 | 构建测试集，自动化评估模型效果 |
| Playground | 在线调试 prompt 和链 |

如果你的项目对数据隐私有要求，也可以考虑开源替代方案（如 LangFuse、Phoenix 等）。

---

## 8. 部署简介

Notebook 里跑通了，怎么让别人通过 HTTP 接口调用你的链？

> **注意**：LangChain 曾提供 LangServe 库来一键部署 LCEL 链，但该项目已停止维护。官方推荐迁移到 **LangGraph Platform**，或者直接用 **FastAPI** 手动包装。
>
> 下面演示最通用的 FastAPI 方案——不依赖任何额外框架，完全可控。

In [ ]:
# 以下是 FastAPI 部署示例（保存为 .py 文件使用，不在 Notebook 中运行）
server_code = '''
import os
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from fastapi.responses import StreamingResponse

# 1. 定义链
llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)
chain = ChatPromptTemplate.from_template("翻译成英语：{text}") | llm | StrOutputParser()

# 2. 定义请求模型
class TranslateRequest(BaseModel):
    text: str

# 3. 创建 FastAPI 应用
app = FastAPI(title="翻译服务")

@app.post("/translate")
async def translate(req: TranslateRequest):
    """同步调用"""
    result = await chain.ainvoke({"text": req.text})
    return {"result": result}

@app.post("/translate/stream")
async def translate_stream(req: TranslateRequest):
    """流式调用"""
    async def generate():
        async for chunk in chain.astream({"text": req.text}):
            yield chunk

    return StreamingResponse(generate(), media_type="text/plain")

# 启动：uvicorn server:app --host 0.0.0.0 --port 8000
'''
print(server_code)

直接用 FastAPI 包装 LCEL 链，好处是完全可控，不依赖额外抽象层：

- `chain.ainvoke()` 用于异步同步调用
- `chain.astream()` 用于流式输出
- 请求/响应格式由你的 Pydantic model 定义

```bash
# 安装
uv add fastapi uvicorn

# 启动
uvicorn server:app --host 0.0.0.0 --port 8000
```

部署到生产环境时，还需考虑：鉴权（API Key / JWT）、HTTPS、容器化（Docker）、负载均衡、日志收集等。这些是通用的 Web 服务部署问题，不是 LangChain 特有的。

### 其他部署选项

| 方案 | 说明 |
|------|------|
| **FastAPI**（推荐） | 通用、灵活，直接调用 LCEL 链的 `ainvoke` / `astream` |
| **LangGraph Platform** | LangChain 官方托管平台，适合复杂 Agent 工作流 |
| ~~LangServe~~ | 已停止维护，不建议新项目使用 |

---

## 9. 学习路线回顾

七章学完了，回顾一下完整的学习路径：

```
第一章 环境搭建        基石：Models、Runnable 接口
  |
第二章 Prompt Templates  学会构建和复用提示词
  |
第三章 LCEL 链式表达式   用管道符组合组件，掌握数据流
  |
第四章 Memory           让对话有上下文记忆
  |
第五章 RAG              连接外部知识，解决幻觉问题
  |
第六章 Agents           让 LLM 自主决策、调用工具
  |
第七章 生产实践          调试、监控、缓存、部署
```

每个技能点对应可以构建的实际应用：

| 技能 | 可以构建的应用 |
|------|---------------|
| Prompt + LCEL | 翻译工具、文本分析、格式转换 |
| Memory | 聊天机器人、客服系统 |
| RAG | 企业知识库、文档问答 |
| Agent | 智能助手、自动化工作流 |
| 生产实践 | 可靠、可观测的线上服务 |

### 进阶方向

- **LangGraph**：构建更复杂的多步骤、有状态的 Agent 工作流（LangChain 团队的下一代 Agent 框架）
- **LangSmith 深入**：评估体系、数据集管理、A/B 测试
- **多模态**：图像、音频、视频的理解与生成
- **更多向量数据库**：Chroma、Pinecone、Weaviate 等生产级选择
- **微调与优化**：针对特定任务微调模型，优化推理成本

---

## 10. 总结

### 本章学到的

- [x] `set_verbose` / `set_debug`：两级调试开关
- [x] `get_graph().print_ascii()`：可视化链的结构
- [x] Callback 系统：自定义处理器，精确监控调用过程
- [x] `with_structured_output`：利用 Pydantic 获得类型安全的结构化输出
- [x] `InMemoryCache` / `SQLiteCache`：缓存减少重复调用
- [x] `with_retry`：自动重试，指数退避
- [x] `with_fallbacks`：主备模型降级
- [x] `InMemoryRateLimiter`：客户端速率限制
- [x] LangSmith：零侵入的可观测性追踪
- [x] FastAPI 部署：手动包装 LCEL 链为 REST API

### 速查表

| 功能 | API | 说明 |
|------|-----|------|
| 调试-简略 | `set_verbose(True)` | 打印链的输入输出 |
| 调试-详细 | `set_debug(True)` | 打印所有中间状态 |
| 链结构 | `chain.get_graph().print_ascii()` | ASCII 结构图 |
| 回调 | `config={"callbacks": [handler]}` | 自定义监控逻辑 |
| 结构化输出 | `llm.with_structured_output(Model)` | Pydantic model 约束输出 |
| 内存缓存 | `set_llm_cache(InMemoryCache())` | 进程内缓存 |
| 持久缓存 | `set_llm_cache(SQLiteCache(...))` | 文件级持久缓存 |
| 自动重试 | `llm.with_retry(...)` | 指数退避重试 |
| 降级 | `llm.with_fallbacks([backup])` | 主备切换 |
| 限速 | `InMemoryRateLimiter(...)` | 令牌桶限流 |
| 追踪 | 设置 `LANGCHAIN_TRACING_V2` 环境变量 | LangSmith 零侵入追踪 |
| 部署 | FastAPI + `chain.ainvoke()` | 手动包装为 REST API |

---

恭喜完成全部七章的学习！你已经掌握了 LangChain 的核心功能，具备独立构建 LLM 应用的能力。接下来，选择一个你感兴趣的项目，把学到的知识用起来吧。